In [3]:
import sys
sys.path.append('..')
from src.database import init_db

# Esto crea el archivo .db y las tablas si no existen
init_db()
print("✅ Tablas 'movimientos' y 'reglas' creadas con éxito.")

✅ Tablas 'movimientos' y 'reglas' creadas con éxito.


In [4]:
import sys
import os
import glob
sys.path.append('..')
from src.processor import aplicar_reglas, procesar_excel_brou

# Buscar archivos .xls o .xlsx
files = glob.glob("../data/raw/*.xls*") 

if not files:
    print("❌ No se encontraron archivos Excel en data/raw/")
else:
    latest_file = max(files, key=os.path.getctime)
    print(f"Procesando: {latest_file}")
    
    # Cargar y procesar
    df_nuevo = procesar_excel_brou(latest_file)
    
    # Ahora aplicar_reglas ya encontrará la tabla 'reglas' (aunque esté vacía)
    df_con_reglas = aplicar_reglas(df_nuevo)

    # Filtrar pendientes
    pendientes = df_con_reglas[df_con_reglas['categoria'].isna()]

    if pendientes.empty:
        print("✅ ¡Excelente! Todos los movimientos tienen categoría.")
        listo_para_guardar = True
    else:
        print(f"⚠️ Hay {len(pendientes)} movimientos sin categoría.")
        listo_para_guardar = False
        display(pendientes.head())

Procesando: ../data/raw/Detalle_Movimiento_Cuenta.xls
--- DEBUG: Iniciando lectura de ../data/raw/Detalle_Movimiento_Cuenta.xls ---
--- DEBUG: ¡Tabla real encontrada en fila 31! ---
--- DEBUG: Columnas finales detectadas: ['Fecha', 'Descripción', 'Unnamed: 2', 'Número de documento', 'Asunto', 'Dependencia', 'Débito', 'Crédito']
--- DEBUG: Se procesaron 21 movimientos ---
⚠️ Hay 21 movimientos sin categoría.


,fecha,descripcion,monto,hash,categoria
0,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Envio,-70.82,7461135742591152002,None
1,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Locos,-1287.46,14289839871079085353,None
2,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Propin,-30.00,15148875007829001908,None
3,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Envio,-48.20,11067871429215834953,None
4,2026-05-04 00:00:00,Comercio: DLO*PedidosYa Sandwi,-287.13,11045368651333818264,None


In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
# Asegúrate de importar tu conexión a la base de datos
from src.database import get_connection 

pendientes = df_con_reglas[df_con_reglas['categoria'].isnull()]
categorias = ["Super", "Fijos", "Ocio", "Transporte", "Salud", "Educación", "Delivery"]

if not pendientes.empty:
    # Usamos .index[0] para obtener la posición real en el DataFrame original
    idx = pendientes.index[0]
    row = pendientes.iloc[0]
    
    print(f"Gasto desconocido: {row['descripcion']} - Monto: {row['monto']}")
    selector = widgets.Dropdown(options=categorias, description='Categoría:')
    boton = widgets.Button(description="Guardar Regla", button_style='warning')
    
    def guardar_y_seguir(b):
        conn = get_connection()
        # Tomamos una parte de la descripción como palabra clave para futuras coincidencias
        keyword = str(row['descripcion'])[:15].strip() 
        
        try:
            # Usamos INSERT OR IGNORE por si la palabra clave ya existe
            conn.execute("INSERT OR IGNORE INTO reglas (keyword, categoria) VALUES (?, ?)", 
                         (keyword, selector.value))
            conn.commit()
            conn.close()
            clear_output()
            print(f"✅ Regla guardada: {keyword} -> {selector.value}")
            print("🔄 Ejecuta la celda de procesamiento nuevamente para actualizar los pendientes.")
        except Exception as e:
            print(f"❌ Error al guardar en la base de datos: {e}")
            conn.close()
        
    boton.on_click(guardar_y_seguir)
    display(selector, boton)
else:
    print("✅ Todo categorizado. Listo para guardar en BD.")

Gasto desconocido: Comercio: DLO*PedidosYa Envio - Monto: -70.82


Dropdown(description='Categoría:', options=('Super', 'Fijos', 'Ocio', 'Transporte', 'Salud', 'Educación', 'Del…

Button(button_style='warning', description='Guardar Regla', style=ButtonStyle())

In [8]:
import pandas as pd
import plotly.express as px
from src.database import get_connection

# Guardar en base de datos (ignora duplicados por el hash)
conn = get_connection()

try:
    # 'if_exists=append' junto con el índice UNIQUE en la DB evita duplicados
    df_con_reglas.to_sql("movimientos", conn, if_exists="append", index=False, method="multi")
    print("💾 Datos guardados exitosamente.")
except Exception as e:
    print(f"Nota: Algunos registros ya existían y se omitieron.")

# Cargar todo para el gráfico
# Aquí es donde fallaba porque faltaba 'import pandas as pd'
df_total = pd.read_sql("SELECT * FROM movimientos", conn)

# Filtramos solo los gastos para que el gráfico de torta sea coherente
df_gastos = df_total[df_total['monto'] < 0].copy()
df_gastos['monto'] = df_gastos['monto'].abs()

if not df_gastos.empty:
    fig = px.pie(df_gastos, values='monto', names='categoria', title='Mis Gastos Totales')
    fig.show()
else:
    print("No hay gastos registrados para mostrar.")

conn.close()

Nota: Algunos registros ya existían y se omitieron.
